# FlashAttention：Kernel 级 Tiling + Online Softmax

> Dao et al., 2022. 核心考点：分块矩阵乘 + online softmax + HBM/SRAM IO 分析。

## 背景
标准 Attention 需物化 N*N 矩阵 S，FlashAttention 用 tiling 把 Q/K/V 分块加载到 SRAM，
计算局部 attention，用 online softmax 增量更新，从不物化完整 S 矩阵。

## Online Softmax
维护 running max m 和 running sum l：
- m_new = max(m, row_max(block))
- l_new = l * exp(m-m_new) + sum(exp(block-m_new))
- O_new = rescale(O) + softmax(block) @ V_block

## IO 复杂度
标准: O(4N^2 d) HBM 读写。FA: O(N^2 d^2 / M) HBM 读写，M=SRAM(192KB)

## 考察点
- SRAM/HBM 层次与数据搬运
- online softmax 两步变一步推导
- tiling 三重循环：Q沿行切，K/V沿行切


In [ ]:
import torch
import torch.nn.functional as F
import math

def flash_attention(Q, K, V, block_size=32):
    """
    FlashAttention tiling algorithm (简化版)。
    用三重循环将 Q/K/V 分块，每块用 online softmax 避免 O(n^2) HBM 读写。
    Q,K,V: [seq_len, dim], 返回 output [seq_len, dim]
    """
    seq_len, dim = Q.shape
    scale = 1.0 / math.sqrt(dim)
    O = torch.zeros_like(Q)
    L = torch.zeros(seq_len, 1)  # 分母累积
    M = torch.full((seq_len, 1), -float("inf"))  # 当前行最大值

    for b_start in range(0, seq_len, block_size):
        b_end = min(b_start + block_size, seq_len)
        K_block = K[b_start:b_end]   # [Bc, dim]
        V_block = V[b_start:b_end]

        for r_start in range(0, seq_len, block_size):
            r_end = min(r_start + block_size, seq_len)
            Q_block = Q[r_start:r_end]  # [Br, dim]
            O_block = O[r_start:r_end]
            L_block = L[r_start:r_end]
            M_block = M[r_start:r_end]

            S = Q_block @ K_block.T * scale  # [Br, Bc]
            M_new = torch.max(M_block, S.max(dim=1, keepdim=True).values)
            P = torch.exp(S - M_new)  # [Br, Bc]
            L_new = torch.exp(M_block - M_new) * L_block + P.sum(dim=1, keepdim=True)
            O[r_start:r_end] = (torch.exp(M_block - M_new) * O_block + P @ V_block) / L_new
            L[r_start:r_end] = L_new
            M[r_start:r_end] = M_new

    return O

# 验证: 与 naive attention 对比
seq_len, dim = 64, 16
Q = torch.randn(seq_len, dim)
K = torch.randn(seq_len, dim)
V = torch.randn(seq_len, dim)

out_flash = flash_attention(Q, K, V, block_size=16)
scale = 1.0 / math.sqrt(dim)
out_naive = F.softmax(Q @ K.T * scale, dim=-1) @ V

max_err = (out_flash - out_naive).abs().max().item()
assert max_err < 1e-4, f"FlashAttention error too large: {max_err}"
print(f"✅ FlashAttention Kernel: tiling+online softmax 与 naive attention 一致 (max_err={max_err:.2e})")
